# سامانهٔ تشخیص پوششِ صورت — نسخهٔ ۲

نسخهٔ پایه + سه گامِ درخواستی. بقیهٔ الگوریتم **دست‌نخورده** است؛
هر تغییر در کد با `★ گام N` علامت خورده تا پیدا کردنش آسان باشد.

## سه تغییرِ این نسخه

| گام | چه شد | هزینه |
|---|---|---|
| **۵** | قفلِ سبز دیگر دائمی نیست — هر ۲ ثانیه یک بازبینی | ~۱.۷٪ حالتِ عادی |
| **۶** | حداقلِ اندازهٔ صورت + نمایشِ پیشرفتِ بررسی | صفر (کمتر هم می‌شود) |
| **۷** | رنگِ اسکلت = رنگِ وضعیتِ فرد | صفر |

## جدول تصمیم

| رنگ | برچسب | معنی |
|---|---|---|
| ⚪ خاکستری | `Analyzing... (2/3)` | دیده شده، در حالِ بررسی — عدد یعنی چند رأی جمع شده |
| ⚪ خاکستری | `Analyzing... (too far)` | دیده شده ولی هنوز خیلی دور است |
| 🟢 سبز | `Clear` | صورت باز — **هر ۲ ثانیه بازبینی می‌شود** |
| 🟠 نارنجی | `Medical Mask` | ماسک دارد ولی بالای صورت پیداست |
| 🔴 قرمز | `SUSPICIOUS - ALERT` | ماسک دارد و بالای صورت هم پوشیده است |

کادر، اسکلت و برچسب هر سه با همین رنگ کشیده می‌شوند.

> **قبل از شروع:** Runtime ← Change runtime type ← GPU

---
## ۰) نصب و راه‌اندازی

In [ ]:
!pip install -q ultralytics opencv-python-headless transformers torch torchvision pillow

### وارد کردن کتابخانه‌ها و انتخاب دستگاه

In [ ]:
import cv2, time, torch, numpy as np
from collections import deque
from PIL import Image
from ultralytics import YOLO
from transformers import AutoImageProcessor, SiglipForImageClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
USE_HALF = device == "cuda"
print(f"🔧 Device: {device} | FP16: {USE_HALF}")

---
## ۱) مدل ژست — تشخیص فرد، اسکلت و ردیابی

یک مدل، سه کار: جعبهٔ هر فرد، ۱۷ کی‌پوینتِ COCO، و شناسهٔ پایدار برای
دنبال‌کردن هر نفر بین فریم‌ها (ByteTrack).

In [ ]:
# ---------------- 1) Single Unified Model: Person+Pose+Track ----------
pose_model = YOLO("yolo11n-pose.pt")

NOSE, LEYE, REYE, LEAR, REAR = 0, 1, 2, 3, 4
LSHOULDER, RSHOULDER, LELBOW, RELBOW, LWRIST, RWRIST = 5, 6, 7, 8, 9, 10

# اسکلت فقط بالاتنه (سر، شونه، بازو، ساعد) - برای جلوه بصری
UPPER_BODY_SKELETON = [
    (LEYE, REYE), (NOSE, LEYE), (NOSE, REYE),
    (LEAR, LEYE), (REAR, REYE),
    (LSHOULDER, RSHOULDER),
    (LSHOULDER, LELBOW), (LELBOW, LWRIST),
    (RSHOULDER, RELBOW), (RELBOW, RWRIST),
    (NOSE, LSHOULDER), (NOSE, RSHOULDER),
]

---
## ۲) طبقه‌بندِ ماسک

مدلِ دوکلاسهٔ فاین‌تیون‌شده: `mask` یا `no_mask`. دسته‌ای کار می‌کند —
همهٔ صورت‌های یک فریم با هم به مدل می‌روند.

> یک بار این را اجرا کن و با `MASK_ID2LABEL` مقایسه کن:
> `print(mask_model.config.id2label)`

In [ ]:
# ---------------- 2) Mask Classifier (Pretrained, FP16) ----------------
MASK_MODEL_NAME = "prithivMLmods/Face-Mask-Detection"
mask_processor = AutoImageProcessor.from_pretrained(MASK_MODEL_NAME)
mask_model = SiglipForImageClassification.from_pretrained(MASK_MODEL_NAME).to(device).eval()
if USE_HALF:
    mask_model = mask_model.half()
MASK_ID2LABEL = {0: "mask", 1: "no_mask"}

@torch.no_grad()
def classify_mask_batch(face_list):
    if len(face_list) == 0:
        return []
    pil_imgs = [Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in face_list]
    inputs = mask_processor(images=pil_imgs, return_tensors="pt").to(device)
    if USE_HALF:
        inputs = {k: (v.half() if v.dtype == torch.float32 else v) for k, v in inputs.items()}
    logits = mask_model(**inputs).logits.float()
    probs = torch.nn.functional.softmax(logits, dim=1).cpu().numpy()
    out = []
    for p in probs:
        pred = int(np.argmax(p))
        out.append((MASK_ID2LABEL[pred], float(p[pred])))
    return out

---
## ۳) سنجهٔ پوست — تفکیکِ ماسکِ پزشکی از مشکوک

اگر کسی ماسکِ پزشکی زده، بالای صورتش پوست دیده می‌شود. اگر پوششِ
کامل داشته باشد، آنجا هم پوشیده است. `is_suspicious` نسبتِ پوست را
روی ۴۵٪ بالای برش می‌سنجد؛ زیر ۰.۱۲ یعنی مشکوک.

In [ ]:
# ---------------- 3) Skin-ratio heuristic (Medical vs Suspicious) -----
def skin_ratio(region_bgr):
    if region_bgr is None or region_bgr.size == 0:
        return 0.0
    hsv = cv2.cvtColor(region_bgr, cv2.COLOR_BGR2HSV)
    lower = np.array([0, 20, 40], dtype=np.uint8)
    upper = np.array([25, 180, 255], dtype=np.uint8)
    m = cv2.inRange(hsv, lower, upper)
    return float(np.count_nonzero(m)) / m.size

def is_suspicious(face_bgr):
    h, w, _ = face_bgr.shape
    upper = face_bgr[0:int(h * 0.45), :]
    return skin_ratio(upper) < 0.12

---
## ۴) تراز و برشِ صورت  ★ گام ۶

**گیتِ ورودی:** میانگینِ اطمینانِ بینی و دو چشم. زیرِ آستانه → `None`
→ آن فرد در آن فریم رأی نمی‌دهد. همین باعث می‌شود نیم‌رخ‌ها و
پشت‌به‌دوربین‌ها قضاوت نشوند.

**★ گام ۶ — `min_eye_dist`:** آستانهٔ فاصلهٔ دو چشم از ۳ به **۸**
رسید. قبلاً صورتی به عرضِ ~۱۰ پیکسل هم رأی می‌داد و آن رأی نویزِ
خالص بود. حالا فردِ دور همچنان **دیده و ردیابی می‌شود** — فقط تا
نزدیک‌تر نشده حکمی درباره‌اش صادر نمی‌شود.

In [ ]:
# ---------------- 4) Keypoint-based Face Visibility + Align + Crop ----
def align_and_crop_face(person_img, kxy, kconf, conf_th=0.5, min_eye_dist=8):
    nose_c, leye_c, reye_c = kconf[NOSE], kconf[LEYE], kconf[REYE]
    face_score = float((nose_c + leye_c + reye_c) / 3.0)
    if face_score < conf_th:
        return None, face_score

    leye, reye = kxy[LEYE], kxy[REYE]
    eye_dist = float(np.linalg.norm(np.array(leye) - np.array(reye)))
    # ★ گام ۶ — حداقلِ اندازهٔ صورت.
    #   قبلاً این عدد ۳ بود؛ یعنی صورتی به عرضِ ~۱۰ پیکسل هم رأی می‌داد و
    #   آن رأی عملاً نویزِ خالص بود. با ۸، فردِ دور دیده و ردیابی می‌شود
    #   ولی تا وقتی به‌قدرِ کافی نزدیک نشده، حکمی دربارهٔ او صادر نمی‌شود
    #   و در حالتِ «Analyzing» می‌ماند.
    if eye_dist < min_eye_dist:
        return None, face_score

    h, w = person_img.shape[:2]
    dy, dx = reye[1] - leye[1], reye[0] - leye[0]
    angle = np.degrees(np.arctan2(dy, dx))
    eye_center = ((leye[0] + reye[0]) / 2.0, (leye[1] + reye[1]) / 2.0)

    M = cv2.getRotationMatrix2D(eye_center, angle, 1.0)
    rotated = cv2.warpAffine(person_img, M, (w, h))

    half_w = eye_dist * 1.6
    top = eye_center[1] - eye_dist * 1.3
    bottom = eye_center[1] + eye_dist * 2.6

    x1, x2 = int(max(0, eye_center[0] - half_w)), int(min(w, eye_center[0] + half_w))
    y1, y2 = int(max(0, top)), int(min(h, bottom))
    if (x2 - x1) < 10 or (y2 - y1) < 10:
        return None, face_score

    return rotated[y1:y2, x1:x2], face_score

---
## ۵) ماشینِ حالت  ★ گام ۵

رأی‌ها روی چند فریم جمع می‌شوند تا رنگ‌ها چشمک نزنند.

| حالت | نرخِ بررسی |
|---|---|
| `fast` | هر فریم — تا ۳ رأی جمع شود |
| `focus` | هر ۲ فریم — فردِ ماسک‌دار زیرِ نظر |
| `locked` سبز | **هر ۶۰ فریم — ★ گام ۵** |

**★ گام ۵ — شکستنِ قفلِ سبز.** سناریو: کسی با صورتِ باز وارد می‌شود،
سبز قفل می‌شود، بعد داخل ماسک می‌کشد. با قفلِ دائمی سیستم دیگر هرگز
نگاهش نمی‌کرد.

حالا هر ۶۰ فریم (≈۲ ثانیه) یک بار بررسی می‌شود:

- رأی سبز آمد → ساعت صفر می‌شود، سبز می‌ماند
- رأی نارنجی/قرمز آمد → **قفل می‌شکند** و می‌رود به حالتِ فوکوس

**هزینه‌اش دقیقاً چقدر است؟** برای هر فردِ سبز، یک اجرای طبقه‌بند در
هر ۶۰ فریم به‌جای صفر. در حالتِ عادی هر فرد هر فریم بررسی می‌شد،
پس این یعنی **حدودِ ۱.۷٪** آن هزینه. عملاً رایگان.

عدد را می‌توانی عوض کنی: `state_mgr.GREEN_RECHECK_FRAMES = 90`

In [ ]:
# ---------------- 5) Smart State Manager (same logic as v4) -----------
COLORS = {"gray": (160, 160, 160), "green": (0, 200, 0),
          "orange": (0, 140, 255), "red": (0, 0, 255)}
LABELS = {"gray": "Analyzing...", "green": "Clear",
          "orange": "Medical Mask", "red": "SUSPICIOUS - ALERT"}

class TrackStateManager:
    def __init__(self):
        self.data = {}
        self.FAST_VOTES_NEEDED = 3
        self.FOCUS_INTERVAL = 2
        self.FOCUS_WINDOW = 4
        self.FOCUS_GREEN_NEEDED = 3
        # ★ گام ۵ — قفلِ سبز دیگر دائمی نیست.
        #   هر ۶۰ فریم (≈۲ ثانیه در ۳۰fps) یک بار دوباره بررسی می‌شود.
        #   هزینه: برای هر فردِ سبز، یک اجرای طبقه‌بند در هر ۶۰ فریم
        #   به‌جای صفر — یعنی حدودِ ۱.۷٪ حالتِ عادی. عملاً رایگان.
        self.GREEN_RECHECK_FRAMES = 60

    def ensure(self, tid):
        if tid not in self.data:
            self.data[tid] = {
                "mode": "fast", "locked": False, "votes": [],
                "focus_window": deque(maxlen=self.FOCUS_WINDOW),
                "color": "gray", "label": LABELS["gray"],
                "is_new": True,        # برای افکت نمایشی: آیا تازه معرفی شده؟
                "just_finalized": None,# برای افکت نمایشی: آیا همین الان قفل نهایی گرفته؟
                "locked_frame": 0,     # ★ گام ۵: آخرین فریمی که قفل/بازبینی شد
                "checks": 0,           # ★ گام ۶: چند بار تا حالا بررسی شده
            }
        return self.data[tid]

    def should_analyze(self, tid, frame_idx):
        st = self.data[tid]
        if st["locked"]:
            # ★ گام ۵ — بازبینیِ دوره‌ایِ افرادِ سبز.
            #   سناریویی که این را لازم می‌کند: کسی با صورتِ باز وارد
            #   می‌شود، سبز قفل می‌شود، بعد داخلِ مغازه ماسک می‌کشد.
            #   با قفلِ دائمی، سیستم دیگر هرگز نگاهش نمی‌کرد.
            return (frame_idx - st["locked_frame"]) >= self.GREEN_RECHECK_FRAMES
        if st["mode"] == "fast":
            return True
        return frame_idx % self.FOCUS_INTERVAL == 0

    def register_vote(self, tid, category, conf, frame_idx=0):
        st = self.data[tid]
        st["just_finalized"] = None
        st["checks"] += 1                       # ★ گام ۶: شمارشِ بررسی‌ها

        # ★ گام ۵ — رأیِ بازبینی برای فردی که قبلاً سبز قفل شده بود.
        if st["locked"]:
            st["locked_frame"] = frame_idx      # ساعتِ بازبینی صفر شود
            if category == "green":
                return                          # هنوز صورتش باز است — کاری نکن
            # پوشش دیده شد → قفل می‌شکند و می‌رود به حالتِ فوکوس
            st["locked"] = False
            st["mode"] = "focus"
            st["votes"] = []
            st["focus_window"].clear()
            st["focus_window"].append(category)
            st["color"], st["label"] = category, LABELS[category]
            if category == "red":
                st["just_finalized"] = "red"
            return

        if st["mode"] == "fast":
            st["votes"].append((category, conf))
            if len(st["votes"]) >= self.FAST_VOTES_NEEDED:
                score = {"green": 0.0, "orange": 0.0, "red": 0.0}
                for c, cf in st["votes"]:
                    score[c] += cf
                best = max(score, key=score.get)
                if best == "green":
                    self._lock(tid, "green", frame_idx)
                else:
                    st["mode"] = "focus"
                    st["color"], st["label"] = best, LABELS[best]
                    st["focus_window"].append(best)
                    if best == "red":
                        st["just_finalized"] = "red"
        else:
            st["focus_window"].append(category)
            window = list(st["focus_window"])
            green_count = window.count("green")
            if green_count >= self.FOCUS_GREEN_NEEDED:
                self._lock(tid, "green", frame_idx)
            else:
                sub = [c for c in window if c in ("orange", "red")]
                if sub:
                    best = max(set(sub), key=sub.count)
                    if best == "red" and st["color"] != "red":
                        st["just_finalized"] = "red"
                    st["color"], st["label"] = best, LABELS[best]

    def _lock(self, tid, category, frame_idx=0):
        st = self.data[tid]
        st["locked"] = True
        st["locked_frame"] = frame_idx          # ★ گام ۵: شروعِ شمارشِ بازبینی
        st["color"] = category
        st["label"] = LABELS[category]

state_mgr = TrackStateManager()

---
## ۶) ابزارِ نمایش — گالری و اسکلت  ★ گام ۷

**★ گام ۷:** رنگِ پیش‌فرضِ `draw_upper_skeleton` دیگر زردِ ثابت نیست.
حالا فراخوان رنگِ وضعیتِ همان فرد را می‌فرستد، پس کادر و اسکلت و
برچسب هم‌رنگ‌اند.

In [ ]:
# ---------------- 6) Presentation Helpers (Gallery + Skeleton) --------
class PresentationGallery:
    """گالری تصاویر کوچک در گوشه تصویر - آخرین افراد شناسایی‌شده"""
    def __init__(self, max_items=4, thumb_size=140):
        self.items = deque(maxlen=max_items)   # هر آیتم: (img, label, color)
        self.thumb_size = thumb_size

    def add(self, crop_bgr, label, color):
        if crop_bgr is None or crop_bgr.size == 0:
            return
        thumb = cv2.resize(crop_bgr, (self.thumb_size, self.thumb_size))
        self.items.append((thumb, label, color))

    def draw(self, frame):
        h, w = frame.shape[:2]
        pad = 10
        for i, (thumb, label, color) in enumerate(self.items):
            x2 = w - pad
            x1 = x2 - self.thumb_size
            y1 = pad + i * (self.thumb_size + 35)
            y2 = y1 + self.thumb_size
            if y2 > h:
                break
            frame[y1:y2, x1:x2] = thumb
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)
            cv2.putText(frame, label, (x1, y2 + 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

gallery = PresentationGallery(max_items=4, thumb_size=140)

def draw_upper_skeleton(frame, kxy, kconf, color=(160, 160, 160), conf_th=0.4):
    """
    رسم اسکلتِ بالاتنه.

    ★ گام ۷ — رنگِ پیش‌فرض دیگر زردِ ثابت نیست. حالا فراخوان رنگِ
    وضعیتِ همان فرد را می‌فرستد، پس کادر و اسکلت و برچسب هم‌رنگ‌اند
    و کلِ فریم با یک نگاه خوانده می‌شود.
    """
    for a, b in UPPER_BODY_SKELETON:
        if kconf[a] < conf_th or kconf[b] < conf_th:
            continue
        pa = tuple(map(int, kxy[a]))
        pb = tuple(map(int, kxy[b]))
        cv2.line(frame, pa, pb, color, 2)
    for idx in [NOSE, LEYE, REYE, LEAR, REAR, LSHOULDER, RSHOULDER, LELBOW, RELBOW, LWRIST, RWRIST]:
        if kconf[idx] >= conf_th:
            p = tuple(map(int, kxy[idx]))
            cv2.circle(frame, p, 4, color, -1)

---
## ۷) خط لولهٔ اصلی

دو تابعِ تکراریِ نسخهٔ اصلی یکی شده‌اند. بررسی کردم — منطقِ تصمیم در
هر دو **یکسان** بود و نسخهٔ دمو فقط سه چیزِ نمایشی اضافه داشت:

| پرچم | پیش‌فرض | اثر |
|---|---|---|
| `show_skeleton` | `True` | رسمِ اسکلت (حالا هم‌رنگِ وضعیت) |
| `show_gallery` | `True` | گالریِ گوشهٔ تصویر |
| `slowmo_repeat` | `6` | تکرارِ فریم در لحظاتِ کلیدی (`1` = خاموش) |
| `min_eye_dist` | `8` | ★ گام ۶ |

**★ گام ۷ — ترتیبِ رسم عوض شد.** در نسخهٔ اصلی اسکلت در حلقهٔ *اول*
کشیده می‌شد، جایی که هنوز رنگِ وضعیت معلوم نبود. حالا کی‌پوینت‌ها در
`kpts_by_tid` نگه داشته می‌شوند و اسکلت در حلقهٔ *رسم* — بعد از
مشخص‌شدنِ رنگ — کشیده می‌شود.

**★ گام ۶ — برچسبِ پیشرفت.** در حالتِ خاکستری به‌جای
`Analyzing...` خالی، حالا `Analyzing... (2/3)` نوشته می‌شود؛ و اگر
هنوز هیچ بررسی‌ای ممکن نبوده `Analyzing... (too far)`. این‌طور معلوم
است سیستم فرد را دیده و دارد رویش کار می‌کند.

In [ ]:
# ---------------- 7) Main Pipeline ------------------------------------
def process_video(input_path, output_path, conf_thres=0.4, yolo_imgsz=640,
                  face_conf_th=0.5, slowmo_repeat=6,
                  show_skeleton=True, show_gallery=True, min_eye_dist=8):
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        print("❌ Error: Cannot open video")
        return
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
    if not out.isOpened():                                        # [ایمنی]
        raise RuntimeError(f"❌ فایل خروجی باز نشد: {output_path}")
    t0 = time.time()
    last_pct = -1
    frame_idx = 0
    seen_ids = set()   # برای تشخیص "فرد کاملا جدید" جهت افکت اسلوموشن

    results_gen = pose_model.track(
        source=input_path, classes=[0], conf=conf_thres, imgsz=yolo_imgsz,
        tracker="bytetrack.yaml", stream=True, verbose=False, persist=True,
        half=USE_HALF
    )

    for r in results_gen:
        frame_idx += 1
        frame = r.orig_img

        if r.boxes.id is None or r.keypoints is None:
            out.write(frame)
            pct = int(frame_idx / total * 100) if total > 0 else 0
            if pct != last_pct and pct % 5 == 0:
                print(f"⏳ Progress: {pct}%"); last_pct = pct
            continue

        ids = r.boxes.id.int().cpu().tolist()
        boxes = r.boxes.xyxy.cpu().numpy()
        kpts_xy_all = r.keypoints.xy.cpu().numpy()
        kpts_conf_all = r.keypoints.conf.cpu().numpy() if r.keypoints.conf is not None else np.ones(kpts_xy_all.shape[:2])

        batch_crops, batch_meta = [], []
        trigger_slowmo = False   # آیا این فریم باید کند نمایش داده بشه؟
        kpts_by_tid = {}         # ★ گام ۷: برای رسمِ اسکلت با رنگِ وضعیت

        for tid, box, kxy, kconf in zip(ids, boxes, kpts_xy_all, kpts_conf_all):
            x1, y1, x2, y2 = [int(v) for v in box]
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(W, x2), min(H, y2)
            person_crop = frame[y1:y2, x1:x2]
            if person_crop.size == 0:
                continue

            st = state_mgr.ensure(tid)
            kpts_by_tid[tid] = (kxy, kconf)          # ★ گام ۷

            # --- افکت نمایشی: فرد کاملا جدید -> اسلوموشن + اضافه به گالری ---
            if tid not in seen_ids:
                seen_ids.add(tid)
                trigger_slowmo = True
                square_crop = person_crop.copy()
                gallery.add(square_crop, f"New Person ID {tid}", (255, 255, 0))

            if state_mgr.should_analyze(tid, frame_idx):
                local_kxy = kxy.copy()
                local_kxy[:, 0] -= x1
                local_kxy[:, 1] -= y1

                face_crop, face_score = align_and_crop_face(
                    person_crop, local_kxy, kconf, face_conf_th, min_eye_dist)

                if face_crop is not None:
                    batch_crops.append(face_crop)
                    batch_meta.append(tid)

        if batch_crops:
            results = classify_mask_batch(batch_crops)
            for (tid, (mask_label, conf), face_bgr) in zip(batch_meta, results, batch_crops):
                if mask_label == "no_mask":
                    state_mgr.register_vote(tid, "green", conf, frame_idx)
                else:
                    cat = "red" if is_suspicious(face_bgr) else "orange"
                    state_mgr.register_vote(tid, cat, conf, frame_idx)

        # ---- رسم باکس‌ها + تشخیص لحظه هشدار قرمز برای اسلوموشن و گالری ----
        for tid, box in zip(ids, boxes):
            x1, y1, x2, y2 = [int(v) for v in box]
            st = state_mgr.ensure(tid)
            color = COLORS[st["color"]]

            # ★ گام ۷ — اسکلت با رنگِ وضعیتِ همین فرد
            if show_skeleton and tid in kpts_by_tid:
                kxy_d, kconf_d = kpts_by_tid[tid]
                draw_upper_skeleton(frame, kxy_d, kconf_d, color)

            # ★ گام ۶ — در حالتِ «در حال بررسی» پیشرفت را نشان بده تا
            #   معلوم باشد سیستم فرد را دیده و دارد رویش کار می‌کند،
            #   نه اینکه او را نادیده گرفته باشد.
            label = st["label"]
            if st["color"] == "gray":
                got = len(st["votes"])
                if st["checks"] == 0:
                    label = "Analyzing... (too far)"
                else:
                    label = f"Analyzing... ({got}/{state_mgr.FAST_VOTES_NEEDED})"

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, f"ID {tid}: {label}", (x1, max(20, y1 - 8)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

            if st["color"] == "red":
                cv2.putText(frame, "ALERT!", (x1, y2 + 22),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

            if st.get("just_finalized") == "red":
                trigger_slowmo = True
                x1c, y1c = max(0, x1), max(0, y1)
                x2c, y2c = min(W, x2), min(H, y2)
                crop = frame[y1c:y2c, x1c:x2c].copy()
                gallery.add(crop, f"SUSPECT ID {tid}", (0, 0, 255))
                st["just_finalized"] = None  # فقط یکبار افکت رو نشون بده

        # ---- گالری گوشه تصویر ----
        if show_gallery:
            gallery.draw(frame)

        # ---- نوشتن خروجی (با افکت اسلوموشن در لحظات کلیدی) ----
        repeat = slowmo_repeat if trigger_slowmo else 1
        for _ in range(repeat):
            out.write(frame)

        pct = int(frame_idx / total * 100) if total > 0 else 0
        if pct != last_pct and pct % 5 == 0:
            print(f"⏳ Progress: {pct}%")
            last_pct = pct

    out.release()
    print(f"✅ Done in {time.time() - t0:.2f}s -> {output_path}")

---
## ۸) اجرا

### اتصال Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### تنظیم مسیرها و اجرا

In [ ]:
# ---------------- مسیرها ----------------
INPUT_VIDEO  = "/content/drive/MyDrive/9.mp4"     # ← ویدیوی خودت
OUTPUT_VIDEO = "/content/output_v2.mp4"

# ---------------- پارامترها (همان مقادیرِ نسخهٔ اصلی) ----------------
CONF_THRES    = 0.4
YOLO_IMGSZ    = 640
FACE_CONF_TH  = 0.5

# ---------------- ★ گام ۶ ----------------
MIN_EYE_DIST  = 8       # حداقلِ فاصلهٔ دو چشم برای صدورِ حکم

# ---------------- جلوه‌های نمایشی ----------------
SHOW_SKELETON = True
SHOW_GALLERY  = True
SLOWMO_REPEAT = 6       # ۱ = خاموش

import os
if not os.path.exists(INPUT_VIDEO):                               # [ایمنی]
    raise FileNotFoundError(f"❌ ویدیو پیدا نشد: {INPUT_VIDEO}")

# حالتِ داخلی را قبل از هر اجرا صفر می‌کنیم — وگرنه اگر سلول را دو بار
# اجرا کنی، شناسه‌ها و رأی‌های اجرای قبلی باقی می‌مانند.
state_mgr = TrackStateManager()
gallery   = PresentationGallery(max_items=4, thumb_size=140)

# ★ گام ۵ — نرخِ بازبینیِ افرادِ سبز (۶۰ فریم ≈ ۲ ثانیه در ۳۰fps)
state_mgr.GREEN_RECHECK_FRAMES = 60

process_video(INPUT_VIDEO, OUTPUT_VIDEO,
              conf_thres=CONF_THRES,
              yolo_imgsz=YOLO_IMGSZ,
              face_conf_th=FACE_CONF_TH,
              slowmo_repeat=SLOWMO_REPEAT,
              show_skeleton=SHOW_SKELETON,
              show_gallery=SHOW_GALLERY,
              min_eye_dist=MIN_EYE_DIST)

### نمایش ویدیوی خروجی

In [ ]:
import subprocess, os
from base64 import b64encode
from IPython.display import HTML

web = "/content/preview.mp4"
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", OUTPUT_VIDEO,
                "-vcodec", "libx264", "-crf", "26", web], check=False)

path = web if os.path.exists(web) else OUTPUT_VIDEO
data = b64encode(open(path, "rb").read()).decode()
HTML(f'<video width=860 controls>'
     f'<source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

### گزارشِ وضعیتِ نهاییِ هر فرد

بعد از اجرا، این سلول می‌گوید سیستم دربارهٔ هر نفر به چه نتیجه‌ای
رسید و چند بار بررسی‌اش کرد — برای راستی‌آزمایی مفید است.

In [ ]:
from collections import Counter

rows = []
for tid, st in sorted(state_mgr.data.items()):
    rows.append((tid, st["color"], st["checks"], st["locked"], st["mode"]))

print(f"{'ID':>4}  {'وضعیت':<8} {'بررسی':>6} {'قفل':>5}  حالت")
print("-" * 44)
for tid, color, checks, locked, mode in rows:
    print(f"{tid:>4}  {color:<8} {checks:>6} {str(locked):>5}  {mode}")

print("\nجمع‌بندی:", dict(Counter(r[1] for r in rows)))
print("مجموع بررسی‌ها:", sum(r[2] for r in rows))

### ذخیره در Google Drive

In [ ]:
import shutil, os

DRIVE_DEST = '/content/drive/MyDrive/output_v2_saved.mp4'
if os.path.exists(OUTPUT_VIDEO):
    shutil.copy(OUTPUT_VIDEO, DRIVE_DEST)
    print(f"✅ ذخیره شد: {DRIVE_DEST}")
else:
    print("❌ فایل خروجی پیدا نشد — اول سلولِ اجرا را ران کن.")